# Chapter 20 — RAG Pipeline

The final chapter bridges the gap between a trained model and a product people love. Building great AI products requires more than technical excellence — it demands understanding users, designing feedback loops, and measuring value. RAG, personalization, and explainability are your key tools.

In [ ]:
import numpy as np

print("=== RAG (Retrieval-Augmented Generation) ===\n")

documents = [
    "The transformer was introduced in 'Attention Is All You Need' (2017).",
    "BERT uses bidirectional attention for classification tasks.",
    "GPT models use causal attention for text generation.",
    "LoRA adds low-rank matrices to frozen pre-trained weights.",
    "KV-Cache stores key/value tensors to avoid recomputation.",
    "Quantization reduces weights from FP32 to INT4.",
    "RAG retrieves documents to ground LLM responses in facts.",
]

np.random.seed(42)
d_embed = 64
doc_embs = np.random.randn(len(documents), d_embed)
doc_embs /= np.linalg.norm(doc_embs, axis=1, keepdims=True)

def retrieve(question, top_k=3):
    np.random.seed(hash(question) % 2**31)
    q_emb = np.random.randn(d_embed)
    q_emb /= np.linalg.norm(q_emb)
    scores = doc_embs @ q_emb
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(documents[i], scores[i]) for i in top_idx]

question = "How does LoRA work?"
results = retrieve(question)

print(f"Question: \"{question}\"\n")
print("Retrieved documents:")
for rank, (doc, score) in enumerate(results, 1):
    print(f"  [{rank}] (score={score:.4f}) {doc}")

context = "\n".join(f"- {doc}" for doc, _ in results)
print(f"\n=== RAG Prompt ===\nContext:\n{context}\n\nQuestion: {question}\nAnswer:")
print(f"\n(In production: embeddings from sentence-transformers, storage in chromadb)")

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

embedder = SentenceTransformer('all-MiniLM-L6-v2')
db = chromadb.Client()
collection = db.create_collection('knowledge')

collection.add(
    documents=documents,
    embeddings=embedder.encode(documents).tolist(),
    ids=[f'doc_{i}' for i in range(len(documents))],
)

def rag_query(question: str, top_k: int = 3) -> str:
    query_emb = embedder.encode([question]).tolist()
    results = collection.query(query_embeddings=query_emb, n_results=top_k)
    context = '\n'.join(results['documents'][0])
    prompt = f'Context:\n{context}\n\nQuestion: {question}\nAnswer:'
    return model.generate(prompt)

---

**Course:** Aprenda LLM | **Chapter 20**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.